# 🗓 Calendar Agent — Kaggle LoRA Training (HyperCLOVA X SEED 0.5B)

Qwen2.5-0.5B 학습본과 head-to-head 비교용. 같은 학습 데이터·하이퍼파라미터로 베이스 모델만 다르게.

**HyperCLOVA X SEED 0.5B (naver-hyperclovax)**:
- 네이버가 한국어 네이티브로 학습한 모델
- 한국어 시간 표현("저녁 8시", "오전 8시 30분")에서 Qwen 대비 강점 가능성
- `trust_remote_code: true` 필요 (모델 카드 확인됨)
- 라이선스: 사용 전 모델 카드의 라이선스 조건 재확인 필요

## 실행 전 준비 (kaggle_train.ipynb와 동일)

### Kaggle 우측 패널
- **Accelerator**: `GPU T4 x2` 또는 `GPU P100`
- **Internet**: `On`
- **+ Add Input** → Datasets → `calendar-messages` 첨부

### GitHub PAT
https://github.com/settings/tokens/new?type=classic — repo scope.

---

셀을 위에서 아래로 순서대로 실행. 학습 ~1시간.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

In [ ]:
import os, getpass

%cd /kaggle/working
if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/soo-vibe/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, pulling latest…')
    !cd calendar-agent && git pull

%cd /kaggle/working/calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치 + cleanup

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y
import torch
print(f'\ntorch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 3. 데이터 복사 (Kaggle Dataset → 작업 디렉토리)

데이터셋 `calendar-messages`가 `data/` 구조 그대로 업로드된 상태 가정.

In [ ]:
import shutil
from pathlib import Path

DATASET_DIR = Path('/kaggle/input/calendar-messages')
TRAIN = Path('data/processed/v1_train.jsonl')
VAL   = Path('data/processed/v1_val.jsonl')
GOLD  = Path('data/eval/golden.jsonl')
TRAIN.parent.mkdir(parents=True, exist_ok=True)
GOLD.parent.mkdir(parents=True, exist_ok=True)

src_map = [
    (DATASET_DIR / 'data/processed/v1_train.jsonl', TRAIN),
    (DATASET_DIR / 'data/processed/v1_val.jsonl',   VAL),
    (DATASET_DIR / 'data/eval/golden.jsonl',        GOLD),
]

for src, dst in src_map:
    if not src.exists():
        raise FileNotFoundError(f'{src} not found. Dataset 첨부했는지 / 파일 위치 확인.')
    if not dst.exists():
        shutil.copy(src, dst)
        print(f'{src} → {dst}')
    else:
        print(f'{dst} already exists, skipping copy.')

print()
for p in [TRAIN, VAL, GOLD]:
    print(f'✓ {p}' if p.exists() else f'✗ MISSING: {p}')

## 4. 베이스 모델 교체 + sanity check + config 자동 조정

Qwen 노트북과 유일하게 다른 셀:
- `configs/train.yaml`의 `model_config` 라인을 `model_qwen.yaml` → `model_hyperclova.yaml`로 교체
- 나머지(wandb off, fp16 fallback)는 동일

In [ ]:
import orjson, torch
from pathlib import Path

for p in [Path('data/processed/v1_train.jsonl'), Path('data/processed/v1_val.jsonl'), Path('data/eval/golden.jsonl')]:
    assert p.exists(), f'MISSING: {p}'
    n = sum(1 for line in open(p, 'rb') if line.strip())
    print(f'{p}: {n} records')

# 베이스 모델 교체 — Qwen → HyperCLOVA
!sed -i 's|model_config: configs/model_qwen.yaml|model_config: configs/model_hyperclova.yaml|' configs/train.yaml

# 결과 디렉토리도 분리 (Qwen 결과와 충돌 방지)
!sed -i 's|output_dir: models/lora/v1|output_dir: models/lora/hyperclova_v3|' configs/train.yaml
!sed -i 's|run_name: v1_qwen_0.5b_lora|run_name: v3_hyperclova_0.5b_lora|' configs/train.yaml

# wandb off
!sed -i 's/report_to: wandb/report_to: none/' configs/train.yaml

# bf16 -> fp16 if needed (T4 case)
if not torch.cuda.is_bf16_supported():
    !sed -i 's/bf16: true/bf16: false/' configs/train.yaml
    !sed -i 's/fp16: false/fp16: true/' configs/train.yaml
    !sed -i "s|bnb_4bit_compute_dtype: bfloat16|bnb_4bit_compute_dtype: float16|" configs/train.yaml
    print('GPU bf16 미지원 — fp16로 전환')
else:
    print('bf16 사용')

print('\n=== 적용된 config 핵심 라인 ===')
!grep -E 'model_config|output_dir|run_name|report_to|bf16|fp16|bnb_4bit_compute' configs/train.yaml

## 5. 학습 실행 (HyperCLOVA X SEED 0.5B)

- 1970건 × 3 epochs ≈ 165 step
- T4 기준 ~1시간 (모델 사이즈 동일하므로 Qwen과 거의 같은 시간)
- 첫 실행 시 모델 가중치 다운로드 ~1GB (HF Hub에서)

In [ ]:
!python scripts/train_lora.py --config configs/train.yaml

## 6. 결과 패키징 + 다운로드

Output 패널에서 `lora_hyperclova_v3.zip` 다운로드 가능.

In [ ]:
!ls -la models/lora/hyperclova_v3/
!zip -r /kaggle/working/lora_hyperclova_v3.zip models/lora/hyperclova_v3 configs/train.yaml configs/lora.yaml configs/model_hyperclova.yaml
!ls -la /kaggle/working/lora_hyperclova_v3.zip

from IPython.display import FileLink
FileLink('/kaggle/working/lora_hyperclova_v3.zip')

## 다운로드 + 로컬 평가 안내

1. **위 셀의 FileLink 클릭** 또는 **우측 패널 Output > Download**
2. 로컬에서 압축 해제 → `models/lora/hyperclova_v3/`로 복사
3. Merge (base가 HyperCLOVA로 다름):
   ```
   python scripts/merge_lora.py \
     --base naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B \
     --lora models/lora/hyperclova_v3 \
     --out models/merged/hyperclova_v3 \
     --trust_remote_code
   ```
4. 평가:
   ```
   python scripts/eval_model.py \
     --model models/merged/hyperclova_v3 \
     --eval data/eval/golden.jsonl \
     --out logs/eval_hyperclova_v3.json \
     --failures_out data/failures/round_hyperclova_v3.jsonl
   ```
5. **Qwen v3 vs HyperCLOVA v3 비교** — 35건 golden에서 final_score, time_match_rate, has_schedule_acc 직접 비교.